# Chuyên đề 08 · Qwen3-TTS: hai bộ token hóa và một mô hình nói theo luồng: dựng lại các phép tính của báo cáo kỹ thuật

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd08-qwen3-tts.ipynb)

Notebook đi kèm [Chuyên đề 08 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/qwen3-tts/). Chuyên đề 07 kể cách MOSS-TTS dựng dữ liệu và xếp 33
kênh token. Báo cáo kỹ thuật [*Qwen3-TTS*](https://arxiv.org/abs/2601.15621) (Qwen Team, Alibaba, 2026) đặt một câu
hỏi khác: làm sao để mô hình **bắt đầu nói ngay**, khi chữ còn đang đến. Câu trả lời nằm ở bốn chỗ: hai bộ token hóa
(25 Hz một bảng mã; 12,5 Hz mười sáu lớp), một bộ giải mã chạy theo khối, một cách xếp chữ và tiếng thành hai luồng
song song, và một mô-đun MTP sinh các lớp token còn lại ngay trong cùng một khung.

Ta không có hơn 5 triệu giờ tiếng nói hay một mô hình 1,7 tỉ tham số. Nhưng phần lớn thiết kế ấy là **số học viết
rõ trong báo cáo**: bao nhiêu token mỗi giây, khối nào được nhìn khối nào, gói âm thanh đầu tiên ra sau bao lâu. Ta
viết từng phép ấy thành mã, kiểm nó với các bảng của báo cáo, rồi chạy những phần chạy được trên dữ liệu thật: 3.000
bản ghi chữ số tiếng Anh của Chuyên đề 05, và các số đo sẵn của bộ token hóa thu nhỏ ở chuyên đề ấy.

**Bạn sẽ làm:**
1. Đếm hai bộ token hóa: bit mỗi giây, số bước mô hình chính phải chạy cho mỗi giây tiếng nói, số vị trí cho 10 phút
   nói liền; rồi xem trong số đo của Chuyên đề 05 lớp token đầu tiên mang gì.
2. Dựng mặt nạ chú ý theo khối của bộ giải mã 25 Hz, tính gói âm thanh đầu tiên, và bắt một chỗ báo cáo tự mâu thuẫn.
3. Kiểm 12 dòng của Bảng 2 (độ trễ gói đầu = thời gian của mô hình ngôn ngữ + thời gian giải mã), rồi mô phỏng việc
   phát lại theo gói.
4. Xếp một câu đọc chữ số thành hai luồng chữ–tiếng theo đúng quy tắc của mã phát hành, và so với cách xếp tuần tự
   khi chữ đến dần từ một mô hình ngôn ngữ khác.
5. Đo bằng phép đếm vì sao các lớp dư phải được sinh lần lượt trong khung (MTP), đếm bước so với delay pattern của
   Chuyên đề 07, rồi so tỉ số quan trọng của GRPO và GSPO trên một chuỗi 2.000 token.

**Cần biết trước:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/),
[Bài 21 · Khuếch tán & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/diffusion/),
[Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/),
[Chuyên đề 05 · Token hóa âm thanh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/) và
[Chuyên đề 07 · MOSS-TTS](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/moss-tts/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải dữ liệu lần đầu.

Notebook dùng **PyTorch**, như Chuyên đề 05 và 07. Mọi con số của báo cáo được ghi kèm số trang (bản v1 trên arXiv).
Những số lấy từ cấu hình và mã nguồn phát hành ([QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS)) được ghi là
"mã phát hành", vì báo cáo không nêu chúng. Mọi con số khác do các ô bên dưới tính.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

In [ ]:
import hashlib
import io
import json
import math
import os
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 2·10⁻⁴ (0 in ra 0; bỏ “,0” thừa)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)) + 1e-9)
    m = vn(x / 10**e, digits).removesuffix(",0")
    return f"{m}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Phát một đoạn âm thanh 8 kHz ngay trong notebook (giữ nguyên độ to, cắt về [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Dữ liệu: 3.000 chữ số được đọc to, và số đo của Chuyên đề 05

[Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, phiên bản 1.0.10, giấy
phép CC BY-SA 4.0): 6 người, mỗi người đọc mỗi chữ số từ 0 đến 9 năm mươi lần, thu ở 8.000 mẫu mỗi giây, mỗi mẫu
16 bit. Như ở Chuyên đề 05 và 07, ta kiểm MD5 của *nội dung* 3.000 tệp wav. Các bản ghi có số thứ tự từ 0 đến 4 là
tập kiểm tra (300 bản ghi), phần còn lại là tập huấn luyện.

Hai tệp của Chuyên đề 05, kiểm bằng MD5:
- `codec-results.json`: kết quả đo của bộ token hóa thu nhỏ ở chuyên đề ấy (CAT: 12,5 khung mỗi giây, 32 lớp RVQ × 1.024
  mã, 3 seed) khi chỉ giữ k lớp đầu, k = 1 … 32. Huấn luyện nó mất hơn một giờ, nên ta đọc kết quả thay vì chạy lại.
- `codec-weights.json`: ở đây chỉ dùng vài chuỗi token 32 lớp thật mà mô hình đọc nhỏ của Chuyên đề 05 đã sinh ra.

In [ ]:
SR = 8000
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
WORDS = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


R = fetch_json("codec-results.json", "53c34e984a6054d6135be5afd7c9128d")
W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD bị hỏng, hãy xóa và tải lại"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        waves.append(np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768)
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
dur = np.array([len(x) for x in waves]) / SR
TEST, TRAIN = np.where(idx < 5)[0], np.where(idx >= 5)[0]
print(f"{vn(len(waves), 0)} bản ghi, {vn(dur.sum() / 60, 1)} phút; dài {vn(dur.min(), 2)}–{vn(dur.max(), 2)} s")
print(f"kiểm tra {len(TEST)} bản ghi, huấn luyện {vn(len(TRAIN), 0)}; Chuyên đề 05 đo trên {R['split']['test']} bản")
assert len(TEST) == R["split"]["test"] == 300

## 3 · Báo cáo làm gì, và ta dựng lại gì

| Phần | Báo cáo (trang) | Ở đây |
|---|---|---|
| Bộ token hóa 25 Hz | bộ mã hóa của Qwen2-Audio, thêm lớp lấy mẫu lại và VQ ở giữa; giai đoạn 1 học nhận dạng tiếng nói, giai đoạn 2 thêm bộ giải mã mel (tr. 3, Bảng 3 tr. 6) | đếm bit (mục 4) |
| Giải mã 25 Hz | DiT học bằng flow matching, theo khối 8 token, mặt nạ chú ý theo khối; rồi BigVGAN (tr. 3, 6) | mặt nạ, gói đầu (mục 5) |
| Bộ token hóa 12 Hz | 16 lớp × 2.048 mã, lớp 0 ngữ nghĩa (WavLM làm thầy), 15 lớp RVQ, ConvNet nhân quả (tr. 3–4, Bảng 4 tr. 7) | đếm bit; lớp đầu mang gì, qua số đo của Chuyên đề 05 (mục 4) |
| Độ trễ | Bảng 2: thời gian ra token của gói đầu + thời gian giải mã, ở 1, 3 và 6 yêu cầu đồng thời (tr. 5–6) | kiểm 12 dòng, mô phỏng phát lại (mục 6) |
| Hai luồng | chữ và tiếng ghép theo trục kênh; nhận một token chữ là đoán ngay token tiếng (tr. 4) | xếp một câu FSDD theo quy tắc của mã phát hành (mục 7) |
| MTP | mô hình chính đoán lớp 0, mô-đun MTP sinh các lớp dư (tr. 4) | entropy có điều kiện trên một RVQ nhỏ; đếm bước (mục 8) |
| Huấn luyện | tiền huấn luyện S1–S3 (8.192 → 32.768 token); DPO, GSPO, tinh chỉnh theo giọng (tr. 4–5) | độ dài chuỗi (mục 4); tỉ số quan trọng GRPO và GSPO (mục 9) |

Bảng 1 (tr. 2) liệt kê mười mô hình. Mã phát hành (README, tháng 1/2026) chỉ có bộ token hóa 12 Hz và các mô hình
12 Hz; bộ 25 Hz và các mô hình dùng nó chưa được phát hành.

In [ ]:
FAMILY = [  # Bảng 1 (tr. 2): (bộ token hóa, cỡ, biến thể)
    ("12Hz", 1.7, "Base"), ("12Hz", 1.7, "VoiceDesign"), ("12Hz", 1.7, "CustomVoice"),
    ("12Hz", 0.6, "Base"), ("12Hz", 0.6, "CustomVoice"),
    ("25Hz", 1.7, "Base"), ("25Hz", 1.7, "VoiceEditing"), ("25Hz", 1.7, "CustomVoice"),
    ("25Hz", 0.6, "Base"), ("25Hz", 0.6, "CustomVoice"),
]  # fmt: skip
RELEASED = [m for m in FAMILY if m[0] == "12Hz"]  # mã phát hành (README): chỉ các bản 12 Hz
for tok_name in ("12Hz", "25Hz"):
    names = [f"{vn(size, 1)} tỉ {kind}" for t, size, kind in FAMILY if t == tok_name]
    print(f"{tok_name}: {', '.join(names)}")
print(f"{len(FAMILY)} mô hình trong Bảng 1, đã phát hành {len(RELEASED)}")
assert len(FAMILY) == 10 and len(RELEASED) == 5

## 4 · Hai bộ token hóa bằng con số

**Qwen-TTS-Tokenizer-25Hz** (tr. 3) dựng trên bộ mã hóa âm thanh của Qwen2-Audio, một mô hình nhận dạng tiếng nói.
Giai đoạn 1 tiếp tục huấn luyện nó cho nhận dạng tiếng nói (ASR), sau khi chèn vào giữa bộ mã hóa một lớp lấy mẫu lại
và một lớp lượng tử hóa vectơ (VQ): một bảng mã duy nhất 32.768 mã, 25 token mỗi giây (Bảng 3, tr. 6). Giai đoạn 2 gắn
thêm một bộ giải mã tích chập dựng lại phổ mel từ token rồi tinh chỉnh cả mô hình, để token mang thêm chi tiết âm học.

**Qwen-TTS-Tokenizer-12Hz** (tr. 3–4) theo cách tách ngữ nghĩa–âm học của Mimi (bộ token hóa của Moshi): 12,5 khung
mỗi giây, mỗi khung 16 token, mỗi lớp 2.048 mã (Bảng 4, tr. 7). Lớp đầu là lớp ngữ nghĩa, học theo WavLM (WavLM làm
"thầy"); 15 lớp sau là RVQ, mỗi lớp lượng tử hóa phần dư mà các lớp trước chưa tả được. Bộ mã hóa và bộ giải mã đều
hoàn toàn nhân quả, và bộ giải mã chỉ là một ConvNet nhẹ.

Hai cách ấy cho hai mô hình rất khác. Ở bản 25 Hz, mô hình ngôn ngữ chính sinh mọi token. Ở bản 12 Hz, mô hình chính
chỉ sinh lớp 0 của mỗi khung; 15 lớp còn lại do mô-đun MTP sinh (mục 8).

### 🤔 Dự đoán trước
Bộ 12 Hz dùng gần sáu lần số bit mỗi giây của bộ 25 Hz. Vậy mô hình chính của bản nào phải chạy nhiều bước hơn cho
mỗi giây tiếng nói?

In [ ]:
FPS25, CODES25, LAYERS25 = 25, 32768, 1  # tr. 3; Bảng 3, tr. 6
FPS12, CODES12, LAYERS12 = 12.5, 2048, 16  # tr. 3–4; Bảng 4, tr. 7
SR_CODE, HOP_CODE = 24000, 1920  # mã phát hành: 24.000 mẫu mỗi giây, 1.920 mẫu mỗi khung
assert SR_CODE / HOP_CODE == FPS12

TOK = {"25 Hz": (FPS25, LAYERS25, CODES25), "12 Hz": (FPS12, LAYERS12, CODES12)}
INFO = {}
for name, (fps, layers, codes) in TOK.items():
    INFO[name] = {
        "ms mỗi khung": 1000 / fps,
        "bit mỗi token": math.log2(codes),
        "bit mỗi khung": layers * math.log2(codes),
        "bit mỗi giây": fps * layers * math.log2(codes),
        "bước mô hình chính mỗi giây": fps,  # một bước mỗi khung
        "token MTP mỗi giây": fps * (layers - 1),
        "bước mô hình chính cho câu 10 s": 10 * fps,
    }
print(f"{'':32s} {'25 Hz':>8s} {'12 Hz':>8s}")
for key in INFO["25 Hz"]:
    print(f"{key:32s} " + " ".join(f"{vn(INFO[n][key], 1).removesuffix(',0'):>8s}" for n in TOK))
print(f"tỉ số bit mỗi giây: {vn(INFO['12 Hz']['bit mỗi giây'] / INFO['25 Hz']['bit mỗi giây'], 2)}")
print(
    f"so sánh: bộ token hóa của MOSS-TTS (Chuyên đề 07), 12,5 khung × 32 lớp × 10 bit = {vn(12.5 * 32 * 10, 0)} bit/s"
)
assert INFO["25 Hz"]["bit mỗi giây"] == 375 and INFO["12 Hz"]["bit mỗi giây"] == 2200
assert (INFO["25 Hz"]["ms mỗi khung"], INFO["12 Hz"]["ms mỗi khung"]) == (40, 80)
assert INFO["12 Hz"]["token MTP mỗi giây"] == 187.5

Bit mỗi giây và bước mỗi giây là hai chuyện. Bản 25 Hz có ít bit hơn (375 so với 2.200 bit/s), nhưng mô hình chính
phải chạy 25 bước cho mỗi giây tiếng nói; mô hình chính của bản 12 Hz chỉ chạy 12,5 bước, và 187,5 token còn lại mỗi
giây do MTP, một mạng nhỏ hơn nhiều, sinh ra. Báo cáo cho rằng bước thời gian thô hơn của bộ 12 Hz giúp mô hình tự hồi
quy giữ phụ thuộc dài, nên nó đọc đúng chữ hơn (tr. 7). Ô dưới in các tỉ lệ lỗi từ (WER) mà báo cáo đo cho bốn mô hình
gốc, và kết quả trên bài đọc dài.

In [ ]:
SEED = {  # Bảng 5 (tr. 8): WER (%) trên bộ đề Seed-TTS, (test-zh, test-en)
    "25Hz-0.6B": (1.18, 1.64), "25Hz-1.7B": (1.10, 1.49), "12Hz-0.6B": (0.92, 1.32), "12Hz-1.7B": (0.77, 1.24),
}  # fmt: skip
LONG = {"25Hz-1.7B": (1.517, 1.225), "12Hz-1.7B": (2.356, 2.812)}  # Bảng 10 (tr. 11): (long-zh, long-en)
for size in ("0.6B", "1.7B"):
    a, b = SEED[f"25Hz-{size}"], SEED[f"12Hz-{size}"]
    print(f"Seed-TTS, {size}: 25 Hz {vn(a[0])} | {vn(a[1])} · 12 Hz {vn(b[0])} | {vn(b[1])}")
    assert b[0] < a[0] and b[1] < a[1], "bản 12 Hz thắng bản 25 Hz cùng cỡ ở cả hai tập"
a, b = LONG["25Hz-1.7B"], LONG["12Hz-1.7B"]
print(f"bài đọc dài trên 10 phút, 1.7B: 25 Hz {vn(a[0], 3)} | {vn(a[1], 3)} · 12 Hz {vn(b[0], 3)} | {vn(b[1], 3)}")
print(f"(phần lời ở tr. 11 ghi cho bản 25 Hz: {vn(1.533, 3)} | {vn(1.571, 3)})")
assert a[0] < b[0] and a[1] < b[1], "ở bài đọc dài, bản 25 Hz thắng"

Ở bộ đề Seed-TTS, bản 12 Hz thắng bản 25 Hz cùng cỡ ở cả hai tập, và bản 12 Hz 1,7 tỉ đạt WER 1,24 trên test-en, thấp
nhất trong Bảng 5 (tr. 7–8). Nhưng ở bài đọc dài, bản 25 Hz lại thắng; báo cáo đoán rằng token ngữ nghĩa giúp giữ ổn
định trên chuỗi dài (tr. 11). (Phần lời ở tr. 11 ghi WER của bản 25 Hz là 1,533 và 1,571, khác với Bảng 10.)

**Bao nhiêu vị trí cho 10 phút nói liền?** Báo cáo nói mô hình đọc được hơn 10 phút liền một mạch (tr. 2, 11), không
chia đoạn như các hệ thống ghép từng khúc. Giai đoạn tiền huấn luyện thứ ba (S3) mở độ dài chuỗi tối đa từ 8.192 lên
32.768 token (tr. 5). Mỗi khung là một vị trí của mô hình chính (ở cách xếp hai luồng, mục 7, chữ đi chung vị trí với
tiếng nên không tốn thêm).

In [ ]:
SEQ_S12, SEQ_S3 = 8192, 32768  # độ dài chuỗi tối đa trước và trong giai đoạn S3 (tr. 5)
LONG_MIN = 10  # "hơn 10 phút" tiếng nói (tr. 2, 11)
for name in TOK:
    fps = INFO[name]["bước mô hình chính mỗi giây"]
    print(f"{name}: 10 phút = {vn(LONG_MIN * 60 * fps, 0):>6s} vị trí · {vn(SEQ_S12, 0)} vị trí = "
          f"{vn(SEQ_S12 / fps / 60, 1):>4s} phút · {vn(SEQ_S3, 0)} vị trí = {vn(SEQ_S3 / fps / 60, 1)} phút")  # fmt: skip
assert LONG_MIN * 60 * FPS12 == 7500 < SEQ_S12 < LONG_MIN * 60 * FPS25 == 15000 < SEQ_S3

Với bản 25 Hz, 10 phút là 15.000 vị trí, gần gấp đôi 8.192: chỉ sau S3 nó mới vừa một chuỗi. Bản 12 Hz vừa sẵn trong
8.192 vị trí (khoảng 10,9 phút), và 32.768 vị trí là khoảng 43,7 phút. Chưa tính lời dẫn, chỉ dẫn điều khiển và giọng
mẫu, những thứ cũng chiếm vị trí.

### Lớp đầu mang gì: số đo của Chuyên đề 05

Báo cáo nói lớp đầu của bộ 12 Hz mang nội dung, các lớp sau mang chi tiết âm học (tr. 2–3), nhưng không đo riêng từng
lớp. Chuyên đề 05 đã đo đúng câu hỏi này trên bộ token hóa thu nhỏ của nó: giải mã chỉ từ k lớp đầu, rồi để hai
"người chấm" nghe lại, một CNN đoán chữ số (nội dung) và một CNN đoán người đọc (giọng). Bộ ấy cũng có một mất mát
ngữ nghĩa (λ_sem = 20): một mô hình nhỏ đọc token và chép ra chữ số. Đó là giám sát ngữ nghĩa kiểu ASR, gần với giai
đoạn 1 của bộ 25 Hz hơn là thầy WavLM của bộ 12 Hz. Chuyên đề 05 còn huấn luyện một bản không có mất mát ấy. Ô dưới
đọc cả hai, trung bình 3 seed trên 300 bản ghi kiểm tra.

In [ ]:
KS = np.array(R["ks"])  # số lớp giữ lại: 1, 2, 4, …, 32


def per_layer(name, key):
    """(3 seed, 9 mức lớp): số đo của Chuyên đề 05 trên 300 bản ghi kiểm tra."""
    return np.array([s["test"][key] for s in R["codec"][name]])


ACC = {(n, key): per_layer(n, key) for n in ("ref", "sem0") for key in ("digit", "spk", "mel")}
print(" lớp |  bit/s | chữ số | giọng  | không ngữ nghĩa: chữ số | giọng")
for i, k in enumerate(KS):
    a = [ACC[n, key][:, i].mean() for n in ("ref", "sem0") for key in ("digit", "spk")]
    print(f"{k:4d} | {vn(12.5 * k * 10, 0):>6s} | {pct(a[0]):>6s} | {pct(a[1]):>6s} | {pct(a[2]):>23s} | {pct(a[3])}")
d, s_ = ACC["ref", "digit"].mean(0), ACC["ref", "spk"].mean(0)
i4 = list(KS).index(4)
print(f"\nmột lớp: chữ số {pct(d[0])} (32 lớp: {pct(d[-1])}), giọng {pct(s_[0])} (32 lớp: {pct(s_[-1])})")
print(
    f"từ 1 lên 4 lớp: chữ số tăng {vn(100 * (d[i4] - d[0]), 1)} điểm, giọng tăng {vn(100 * (s_[i4] - s_[0]), 1)} điểm"
)
for key, label in (("digit", "chữ số"), ("spk", "giọng")):
    r1, z1 = ACC["ref", key][:, 0], ACC["sem0", key][:, 0]
    print(f"một lớp, {label}: có mất mát ngữ nghĩa {pct(r1.min())}–{pct(r1.max())}, "
          f"không có {pct(z1.min())}–{pct(z1.max())} (3 seed)")  # fmt: skip
mel_r, mel_z = ACC["ref", "mel"][:, -1].mean(), ACC["sem0", "mel"][:, -1].mean()
print(f"khoảng cách mel ở 32 lớp: có mất mát ngữ nghĩa {vn(mel_r)}, không có {vn(mel_z)} (thấp hơn là tốt hơn)")
print(
    f"người chấm trên bản ghi gốc: chữ số {pct(R['judges']['digit']['test'])}, giọng {pct(R['judges']['spk']['test'])}"
)
assert s_[i4] - s_[0] > 2 * (d[i4] - d[0]), "giọng được lợi từ các lớp dư nhiều hơn nội dung"
assert ACC["sem0", "digit"][:, 0].max() < ACC["ref", "digit"][:, 0].min(), "dải 3 seed không chồng nhau"
assert ACC["sem0", "spk"][:, 0].min() < ACC["ref", "spk"][:, 0].max(), "giọng ở 1 lớp: dải 3 seed chồng nhau"

Hình dưới vẽ hai người chấm theo số lớp giữ lại (trục ngang theo thang log₂): xanh dương tròn là chữ số, cam vuông là
giọng; nét liền là bộ có mất mát ngữ nghĩa, nét đứt rỗng ruột là bộ không có. Dải mờ là khoảng từ seed thấp nhất đến
seed cao nhất.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
xk = np.log2(KS)
for key, col, mk, label in (("digit", BLUE, "o", "chữ số"), ("spk", ORANGE, "s", "giọng")):
    for n, ls, fill, tag in (("ref", "-", col, "có mất mát ngữ nghĩa"), ("sem0", "--", "white", "không có")):
        v = 100 * ACC[n, key]
        ax.fill_between(xk, v.min(0), v.max(0), color=col, alpha=0.15, lw=0)
        ax.plot(xk, v.mean(0), ls, color=col, marker=mk, mfc=fill, ms=6, label=f"{label} · {tag}")
ax.set_xticks(xk, [str(k) for k in KS])
ax.set(xlabel="số lớp token giữ lại (k)", ylabel="người chấm đoán đúng (%)", ylim=(65, 95))
ax.legend(fontsize=8, frameon=False, loc="lower right", ncol=2)
vn_axes(fig)
plt.tight_layout()
plt.show()

Một lớp đã đủ cho người chấm nghe đúng chữ số 81,9% số lần, gần 90,3% của đủ 32 lớp; giọng thì mới 72,0%. Từ 1 lên
4 lớp, giọng tăng 18,6 điểm còn chữ số chỉ 7,0 điểm. Bỏ mất mát ngữ nghĩa, lớp đầu mang ít nội dung hẳn (71,9%, dải 3
seed không chồng nhau), còn giọng ở lớp đầu thay đổi trong dao động giữa các seed. Đó đúng là hình dạng báo cáo mô tả:
lớp đầu giữ "nói gì", các lớp dư thêm "ai nói, nói thế nào". Nhưng ranh giới không sạch: một lớp đã nghe ra giọng
tốt hơn hẳn đoán mò (1/6), và cái giá của giám sát ngữ nghĩa là âm thanh kém đi một chút ở mọi mức (khoảng cách mel ở
32 lớp 1,59 so với 1,48).

Bảng 3 của báo cáo là cùng đánh đổi ấy nhìn từ phía kia: khi giai đoạn 2 thêm chi tiết âm học vào token 25 Hz, mô hình
nhận dạng đọc token kém đi một chút ở cả bốn cột. Bảng 4 so bộ 12 Hz với Mimi, cùng 16 lớp × 2.048 mã × 12,5 khung/s,
trên 2.620 câu của LibriSpeech test-clean (tr. 6–7).

In [ ]:
ASR_WER = {  # Bảng 3 (tr. 6): WER (%) khi nhận dạng từ token 25 Hz; C.V. EN, C.V. CN, Fleurs EN, Fleurs CN
    "giai đoạn 1": (7.51, 10.73, 3.07, 4.23),
    "giai đoạn 2": (10.40, 14.99, 4.14, 4.67),
}
COLS = ("C.V. EN", "C.V. CN", "Fleurs EN", "Fleurs CN")
for c, w1, w2 in zip(COLS, *ASR_WER.values(), strict=True):
    print(f"{c:9s}: giai đoạn 1 {vn(w1):>5s} → giai đoạn 2 {vn(w2):>5s}  (+{vn(w2 - w1)} điểm)")
assert all(b > a for a, b in zip(*ASR_WER.values(), strict=True)), "giai đoạn 2 tăng WER ở mọi cột"

RECON = {  # Bảng 4 (tr. 7): PESQ-WB, PESQ-NB, STOI, UTMOS, SIM (cao hơn là tốt hơn)
    "Mimi": (2.88, 3.42, 0.94, 3.87, 0.87),
    "Qwen 12 Hz": (3.21, 3.68, 0.96, 4.16, 0.95),
}
METRICS = ("PESQ-WB", "PESQ-NB", "STOI", "UTMOS", "SIM")
print(f"\nMimi → Qwen 12 Hz trên {vn(2620, 0)} câu LibriSpeech test-clean:")
print(" · ".join(f"{m} {vn(a)} → {vn(b)}" for m, a, b in zip(METRICS, *RECON.values(), strict=True)))
assert all(b > a for a, b in zip(*RECON.values(), strict=True))

## 5 · Giải mã 25 Hz theo khối

Token 25 Hz không tự thành âm thanh. Bộ giải mã có hai tầng (tr. 3): một Transformer khuếch tán (DiT), học bằng flow
matching, biến token thành phổ mel; rồi một BigVGAN đã sửa biến phổ mel thành sóng âm. Để chạy theo luồng, token được
gom thành các khối 8 token (tr. 6), và DiT dùng một **mặt nạ chú ý theo khối**: token trong khối i chỉ được nhìn các
khối từ i − 3 đến i + 1, tức 3 khối nhìn lại, khối hiện tại và 1 khối nhìn trước (tr. 3):

$$M_{ik} = \mathbb{1}\big[\, i-3 \le k \le i+1 \,\big]$$

Khối nhìn trước làm chất lượng tốt hơn, nhưng có giá: phổ mel của khối 0 chưa tính được khi khối 1 chưa có. BigVGAN
cũng cần thêm 130 ms phổ mel phía sau mới ra được sóng âm của một đoạn (tr. 6).

### 🤔 Dự đoán trước
Mô hình ngôn ngữ phải sinh bao nhiêu token trước khi có gói âm thanh đầu tiên? Gói ấy chứa bao nhiêu ms âm thanh?

In [ ]:
CHUNK, BACK, AHEAD = 8, 3, 1  # token mỗi khối (tr. 6); số khối nhìn lại và nhìn trước (tr. 3)
MS25, VOC_MS = 1000 / FPS25, 130  # 40 ms mỗi token; phần mel phía sau mà BigVGAN cần (tr. 6)


def block_mask(n_tok, chunk=CHUNK, back=BACK, ahead=AHEAD):
    """M[a, b] = True khi token a được nhìn token b: khối của b nằm trong [khối(a) − back, khối(a) + ahead]."""
    blk = torch.arange(n_tok) // chunk
    return (blk[None, :] >= blk[:, None] - back) & (blk[None, :] <= blk[:, None] + ahead)


def need_tokens(i, chunk=CHUNK, back=BACK, ahead=AHEAD):
    """Số token phải có trước khi DiT tính được phổ mel của khối i: token cuối mà khối i nhìn tới, cộng 1."""
    M = block_mask((i + ahead + 2) * chunk, chunk, back, ahead)
    return int(M[i * chunk : (i + 1) * chunk].nonzero()[:, 1].max()) + 1


M64 = block_mask(64)
blk = torch.arange(64) // CHUNK
sees5 = sorted(set(blk[M64[5 * CHUNK]].tolist()))
print(f"mặt nạ {tuple(M64.shape)}; khối 5 nhìn các khối {sees5}")
for i in range(4):
    mel_end = (i + 1) * CHUNK * MS25  # phổ mel đã có đến hết khối i
    print(f"khối {i}: cần {need_tokens(i):2d} token → mel đến {vn(mel_end, 0):>5s} ms → "
          f"âm thanh đến {vn(mel_end - VOC_MS, 0):>5s} ms")  # fmt: skip
wait, first_mel = need_tokens(0), CHUNK * MS25
print(f"gói đầu: chờ {wait} token, có {vn(first_mel, 0)} ms phổ mel, ra {vn(first_mel - VOC_MS, 0)} ms âm thanh; "
      f"sau đó cứ {CHUNK} token một gói {vn(first_mel, 0)} ms")  # fmt: skip
assert (wait, first_mel, first_mel - VOC_MS) == (16, 320, 190), "khác với tr. 6 của báo cáo"
assert [need_tokens(i) - need_tokens(i - 1) for i in range(1, 4)] == [CHUNK] * 3

Mô hình phải sinh 16 token (640 ms nội dung) trước gói đầu, và gói ấy chỉ chứa 190 ms âm thanh: đúng các số ở tr. 6.
Sau đó cứ mỗi 8 token là một gói 320 ms.

**Một chỗ báo cáo tự mâu thuẫn.** Tr. 3 viết vùng nhìn của DiT "giới hạn trong 4 khối: khối hiện tại, 3 khối nhìn lại
và 1 khối nhìn trước". Danh sách ấy là 3 + 1 + 1 = 5 khối, và mặt nạ ở trên cho khối 5 nhìn đúng 5 khối. Con số 16
token ở tr. 6 khớp với 1 khối nhìn trước, nhưng không cho biết số khối nhìn lại: ô dưới đổi số khối nhìn lại và thấy
số token phải chờ không đổi.

In [ ]:
PRINTED_BLOCKS = 4  # "restricted to 4 blocks" (tr. 3)
print(f"danh sách ở tr. 3: {BACK} + 1 + {AHEAD} = {len(sees5)} khối; con số ở tr. 3: {PRINTED_BLOCKS} khối")
for back in range(4):
    print(f"  nhìn lại {back} khối, nhìn trước {AHEAD}: vùng nhìn {back + 1 + AHEAD} khối, "
          f"chờ {need_tokens(0, back=back)} token")  # fmt: skip
print(f"  nhìn lại {BACK} khối, không nhìn trước: chờ {need_tokens(0, ahead=0)} token")
assert len(sees5) == BACK + 1 + AHEAD == 5 != PRINTED_BLOCKS
assert {need_tokens(0, back=b) for b in range(4)} == {16} and need_tokens(0, ahead=0) == 8

Hình dưới là mặt nạ cho 64 token (8 khối): ô xanh là cặp (token đang tính, token được nhìn) được phép. Khung cam
đánh dấu các hàng của khối 5: chúng nhìn từ khối 2 đến khối 6, nhưng không nhìn khối 7.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.8))
ax.imshow(M64.numpy(), cmap="Blues", vmin=0, vmax=1.6, interpolation="nearest")
for b in range(1, 8):
    ax.axhline(b * CHUNK - 0.5, color="white", lw=1)
    ax.axvline(b * CHUNK - 0.5, color="white", lw=1)
ax.add_patch(Rectangle((-0.5, 5 * CHUNK - 0.5), 64, CHUNK, fill=False, ec=ORANGE, lw=2))
ax.text(63, 5 * CHUNK + 3.5, "khối 5", color=ORANGE, ha="right", va="center", fontsize=9, fontweight="bold")
ticks = np.arange(8) * CHUNK + CHUNK / 2 - 0.5
ax.set_xticks(ticks, [str(b) for b in range(8)])
ax.set_yticks(ticks, [str(b) for b in range(8)])
ax.set(xlabel="khối của token được nhìn", ylabel="khối của token đang tính")
plt.tight_layout()
plt.show()

**Bộ 12 Hz không chờ ngữ cảnh tương lai.** Bộ giải mã của nó chỉ nhìn về quá khứ, nên mỗi token, tức 80 ms, về
nguyên tắc giải mã được ngay. Để khỏi tốn công lập lịch cho những gói quá nhỏ, báo cáo gom 4 token thành một gói
320 ms (tr. 6). Ô dưới so hai bộ: khi mô hình ngôn ngữ đã sinh token cho bao nhiêu ms nội dung, thì bao nhiêu ms đã
thành âm thanh.

In [ ]:
PKT12 = 4  # token 12 Hz mỗi gói (tr. 6)
n25, n12 = np.arange(0, 65), np.arange(0, 33)
ready25 = np.where(n25 >= wait, (n25 // CHUNK - AHEAD) * first_mel - VOC_MS, 0)  # ms âm thanh đã giải mã
ready12 = (n12 // PKT12) * PKT12 * 80
lag25 = n25 * MS25 - ready25
print(f"25 Hz: token cho {vn(wait * MS25, 0)} ms nội dung → {vn(ready25[wait], 0)} ms âm thanh; "
      f"phần đi trước luôn là {vn(lag25[wait], 0)} ms ở đầu mỗi gói")  # fmt: skip
print(f"12 Hz: token cho {vn(PKT12 * 80, 0)} ms nội dung → {vn(ready12[PKT12], 0)} ms âm thanh")
assert set(lag25[wait::CHUNK]) == {(1 + AHEAD) * first_mel - first_mel + VOC_MS} == {450}

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot([0, 2560], [0, 2560], ":", color=GREY, lw=1, label="không trễ")
ax.step(n25 * MS25, ready25, where="post", color=ORANGE, lw=2, label="25 Hz: DiT theo khối + BigVGAN")
ax.step(n12 * 80, ready12, where="post", color=BLUE, lw=2, ls="--", label="12 Hz: ConvNet nhân quả, gói 4 token")
ax.annotate("", xy=(640, 190), xytext=(640, 640), arrowprops={"arrowstyle": "<->", "color": "k", "lw": 1})
ax.text(670, 400, "450 ms", fontsize=8, va="center")
ax.set(xlabel="nội dung mô hình ngôn ngữ đã sinh token (ms)", ylabel="âm thanh đã giải mã (ms)")
ax.legend(fontsize=8, frameon=False, loc="upper left")
vn_axes(fig)
plt.tight_layout()
plt.show()

Đường cam (25 Hz) nằm dưới đường "không trễ" một khoảng 450 ms ở đầu mỗi gói: 320 ms của khối nhìn trước cộng 130 ms
của BigVGAN. Đường xanh (12 Hz) chạm đường chéo ở mỗi gói. Đây là độ trễ tính theo *nội dung*: mô hình ngôn ngữ phải
sinh trước bao nhiêu. Nó đổi thành bao nhiêu mili giây thật còn tùy mô hình sinh nhanh đến đâu, và đó là Bảng 2.

## 6 · Độ trễ gói đầu: Bảng 2

Báo cáo định nghĩa (tr. 5): **độ trễ gói đầu** = thời gian mô hình ngôn ngữ ra đủ token cho gói đầu (LM TTFP) + thời
gian bộ token hóa giải mã một gói. Cột **LM TPP** là thời gian mô hình ngôn ngữ ra token của một gói ở trạng thái ổn
định, còn **RTF** là thời gian tính chia cho thời lượng âm thanh. Bảng 2 đo bốn mô hình ở 1, 3 và 6 yêu cầu đồng
thời, trên một máy chạy vLLM. Ô dưới chép bảng và kiểm phép cộng ở cả 12 dòng.

In [ ]:
TABLE2 = {  # Bảng 2 (tr. 5): (đồng thời, LM TTFP, giải mã, độ trễ gói đầu, LM TPP, RTF); thời gian theo ms
    "25Hz-1.7B": [(1, 125, 25, 150, 56, 0.253), (3, 222, 62, 284, 64, 0.394), (6, 376, 147, 523, 85, 0.725)],
    "25Hz-0.6B": [(1, 113, 25, 138, 50, 0.234), (3, 198, 62, 260, 59, 0.378), (6, 334, 147, 481, 80, 0.709)],
    "12Hz-1.7B": [(1, 97, 4, 101, 21, 0.313), (3, 190, 5, 195, 24, 0.363), (6, 328, 5, 333, 32, 0.463)],
    "12Hz-0.6B": [(1, 93, 4, 97, 19, 0.288), (3, 174, 5, 179, 22, 0.338), (6, 294, 5, 299, 30, 0.434)],
}
PACKET_MS = 320  # một gói: 8 token 25 Hz hoặc 4 token 12 Hz (tr. 6)
print("mô hình    đồng thời  TTFP + giải mã = gói đầu   LM TPP   RTF")
for name, rows in TABLE2.items():
    for c, ttfp, dec, fpl, tpp, rtf in rows:
        print(f"{name:10s} {c:6d}     {ttfp:4d} + {dec:3d}    = {fpl:4d} ms  {tpp:5d} ms  {vn(rtf, 3)}")
        assert ttfp + dec == fpl, f"{name}, {c} yêu cầu: TTFP + giải mã ≠ độ trễ gói đầu"
        assert tpp < PACKET_MS and rtf < 1
print("12 dòng: TTFP + giải mã = độ trễ gói đầu; mọi LM TPP dưới 320 ms, mọi RTF dưới 1")

Hình dưới chồng hai phần của độ trễ gói đầu: phần dưới (xanh) là thời gian của mô hình ngôn ngữ, phần trên (cam, gạch
chéo) là thời gian giải mã. Mỗi nhóm là một mô hình, ba cột là 1, 3 và 6 yêu cầu đồng thời.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for g, (name, rows) in enumerate(TABLE2.items()):
    for j, (c, ttfp, dec, fpl, _, _) in enumerate(rows):
        x = 4 * g + j
        ax.bar(x, ttfp, color=BLUE, width=0.8, label="mô hình ngôn ngữ (TTFP)" if x == 0 else None)
        ax.bar(x, dec, bottom=ttfp, color=ORANGE, hatch="//", ec="white", width=0.8,
               label="giải mã một gói" if x == 0 else None)  # fmt: skip
        ax.text(x, fpl + 8, str(fpl), ha="center", fontsize=8)
        ax.text(x, -38, f"×{c}", ha="center", fontsize=8, color=GREY)
ax.set_xticks([4 * g + 1 for g in range(4)], list(TABLE2))
ax.tick_params(axis="x", length=0, pad=16)
ax.set(ylabel="độ trễ gói đầu (ms)", ylim=(0, 580))
ax.legend(fontsize=8, frameon=False, loc="upper right")
vn_axes(fig)
plt.tight_layout()
plt.show()

Ở 1 yêu cầu, bản 12 Hz 0,6 tỉ ra gói đầu sau 97 ms (93 + 4), con số trong tóm tắt của báo cáo. Khi số yêu cầu tăng,
thời gian của mô hình ngôn ngữ tăng ở cả bốn mô hình, nhưng phần giải mã thì khác hẳn: DiT và BigVGAN của bản 25 Hz
đi từ 25 lên 147 ms, ConvNet của bản 12 Hz chỉ từ 4 lên 5 ms. Báo cáo gọi bộ giải mã 12 Hz là nhẹ và dễ gộp lô
(tr. 6).

**RTF đo theo đơn vị nào?** Nếu RTF là (LM TPP + giải mã) chia cho 320 ms của một gói, ta có thể tính lại nó. Ô dưới
thử, và thử cả việc chia cho 80 ms, tức một khung 12 Hz.

In [ ]:
print("mô hình    đồng thời  RTF    (TPP+giải mã)/320  (TPP+giải mã)/80")
TOL = 0.0005 + 1e-9  # bảng làm tròn đến 3 chữ số
match320, match80, other = [], [], []
for name, rows in TABLE2.items():
    for c, _, dec, _, tpp, rtf in rows:
        a, b = (tpp + dec) / PACKET_MS, (tpp + dec) / 80
        print(f"{name:10s} {c:6d}     {vn(rtf, 3)}  {vn(a, 4):>17s}  {vn(b, 4):>16s}")
        if abs(rtf - a) <= TOL:
            match320.append(name)
        elif abs(rtf - b) <= TOL:
            match80.append(name)
        else:
            other.append((name, c, rtf - b))
print(f"khớp với /320: {len(match320)} dòng ({', '.join(sorted(set(match320)))}); khớp với /80: {len(match80)} dòng")
for name, c, dev in other:
    print(f"không khớp: {name}, {c} yêu cầu, lệch {vn(dev, 4)} so với /80")
rtf12 = TABLE2["12Hz-1.7B"][0][5]
print(f"theo RTF, bản 12Hz-1.7B ở 1 yêu cầu tốn {vn(rtf12 * PACKET_MS, 0)} ms cho mỗi gói 320 ms")
assert sorted(set(match320)) == ["25Hz-0.6B", "25Hz-1.7B"] and len(match320) == 6
assert len(match80) == 5 and all(n.startswith("12Hz") for n in match80) and len(other) == 1

Sáu dòng 25 Hz khớp đúng (TPP + giải mã)/320. Năm trong sáu dòng 12 Hz thì khớp (TPP + giải mã)/80, như thể cột TPP
của bản 12 Hz là thời gian cho *một khung* chứ không phải một gói 4 khung; dòng còn lại (0,6 tỉ, 6 yêu cầu) lệch
0,0035. Nếu đúng vậy, bản 12 Hz tốn khoảng 100 ms cho mỗi gói 320 ms ở 1 yêu cầu, và đó có thể là lý do RTF của nó
(0,313) cao hơn bản 25 Hz (0,253) dù độ trễ gói đầu thấp hơn. Báo cáo không giải thích chỗ này; đây là cách đọc của
notebook, khớp với các con số chứ không phải điều báo cáo nói.

**Phát lại có phải chờ không?** Gói đầu đến sau độ trễ gói đầu; từ đó mỗi gói tốn RTF × 320 ms (cách đọc chậm nhất:
mô hình ngôn ngữ và bộ giải mã chạy nối tiếp). Gói đầu của bản 25 Hz chỉ có 190 ms âm thanh, các gói sau 320 ms; gói
của bản 12 Hz luôn là 320 ms. Ô dưới mô phỏng 40 gói cho cả 12 dòng và cộng thời gian phải ngừng giữa chừng.

In [ ]:
def schedule(arrive, length):
    """Phát các gói theo thứ tự, mỗi gói khi nó đã đến và gói trước đã phát xong.
    Trả về lúc bắt đầu phát từng gói và tổng thời gian phải ngừng giữa chừng."""
    start, stall, end = np.empty(len(arrive)), 0.0, 0.0
    for k, (a, L) in enumerate(zip(arrive, length, strict=True)):
        start[k] = a if k == 0 else max(a, end)
        stall += start[k] - end if k else 0.0
        end = start[k] + L
    return start, stall


def playback(fpl, first_ms, period, n=40, packet=PACKET_MS):
    """Gói 0 đến lúc fpl, chứa first_ms âm thanh; gói k ≥ 1 đến lúc fpl + k·period, chứa 320 ms."""
    arrive = fpl + period * np.arange(n)
    length = np.r_[first_ms, np.full(n - 1, packet)]
    start, stall = schedule(arrive, length)
    return arrive, start, length, stall


STALL = {}
for name, rows in TABLE2.items():
    first = first_mel - VOC_MS if name.startswith("25") else PACKET_MS
    for c, _, dec, fpl, tpp, rtf in rows:
        STALL[name, c] = playback(fpl, first, rtf * PACKET_MS)[3]
    print(f"{name}: phải ngừng " + ", ".join(f"{vn(STALL[name, c], 0)} ms ở {c} yêu cầu" for c, *_ in rows))
for name in ("25Hz-1.7B", "25Hz-0.6B"):
    c, _, dec, fpl, tpp, rtf = TABLE2[name][2]
    print(f"{name}, 6 yêu cầu: gói thứ hai cần {vn(rtf * PACKET_MS, 0)} ms, gói đầu chỉ phát {vn(first_mel - VOC_MS, 0)} ms; "
          f"nếu chạy gối đầu (mô hình sinh gói sau trong lúc giải mã gói trước): mỗi gói {max(tpp, dec)} ms, "
          f"ngừng {vn(playback(fpl, first_mel - VOC_MS, max(tpp, dec))[3], 0)} ms")  # fmt: skip
assert {k for k, v in STALL.items() if v > 0} == {("25Hz-1.7B", 6), ("25Hz-0.6B", 6)}
assert round(STALL["25Hz-1.7B", 6]) == 232 - 190

Ở trạng thái ổn định, phát lại không bao giờ phải chờ: mọi RTF đều dưới 1, nên mỗi gói được làm xong trước khi gói
trước phát hết. Chỗ duy nhất có thể ngừng là ngay sau gói đầu của bản 25 Hz ở 6 yêu cầu: gói đầu chỉ có 190 ms âm
thanh, mà gói thứ hai cần 232 ms (bản 1,7 tỉ) nếu hai khâu chạy nối tiếp, nên tiếng ngừng 42 ms. Nếu chạy gối đầu thì
không ngừng. Báo cáo không nói hai khâu có chạy gối đầu hay không. Hình dưới vẽ hai trường hợp ở 6 yêu cầu: bậc thang
là âm thanh đã nhận, đường liền là âm thanh đã phát; chỗ đường liền nằm ngang là chỗ phải ngừng.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, name in zip(axes, ("25Hz-1.7B", "12Hz-1.7B"), strict=True):
    c, _, dec, fpl, tpp, rtf = TABLE2[name][2]
    first = first_mel - VOC_MS if name.startswith("25") else PACKET_MS
    arrive, start, length, stall = playback(fpl, first, rtf * PACKET_MS, n=8)
    got = np.cumsum(length)
    ax.step(np.r_[0, arrive], np.r_[0, got], where="post", color=ORANGE, lw=1.8, label="âm thanh đã nhận")
    tt, played = [0, start[0]], [0, 0]
    for s0, L in zip(start, length, strict=True):
        tt += [s0, s0 + L]
        played += [played[-1], played[-1] + L]
    ax.plot(tt, played, color=BLUE, lw=2, label="âm thanh đã phát")
    if stall > 0:  # chỗ ngừng: gói đầu đã phát hết, gói thứ hai chưa đến
        ax.annotate(f"ngừng {vn(stall, 0)} ms", xy=(start[1] - stall / 2, length[0]), xytext=(900, 80),
                    fontsize=8, arrowprops={"arrowstyle": "->", "color": "k", "lw": 0.8})  # fmt: skip
    ax.set(
        title=f"{name}, 6 yêu cầu: ngừng {vn(stall, 0)} ms", xlabel="thời gian từ lúc gửi yêu cầu (ms)", xlim=(0, 2000)
    )
axes[0].set(ylabel="âm thanh (ms)", ylim=(0, 2200))
axes[0].legend(fontsize=8, frameon=False, loc="upper left")
vn_axes(fig)
plt.tight_layout()
plt.show()

## 7 · Hai luồng: chữ và tiếng trên cùng một vị trí

Mô hình chính là một mô hình ngôn ngữ Qwen3. Chữ được tách bằng bộ tách token chuẩn của Qwen, tiếng bằng bộ token hóa
của Qwen-TTS, và một bộ mã hóa người nói được huấn luyện cùng mô hình chính cho vectơ giọng (tr. 4). Báo cáo tả cách
xếp bằng một câu: chữ và tiếng được **ghép theo trục kênh**, và khi nhận một token chữ, mô hình đoán ngay các token
tiếng tương ứng (tr. 4). Mọi đầu vào theo một khuôn hội thoại (ChatML), chỉ dẫn điều khiển đặt ở đầu (tr. 4–5).

Mã phát hành (`modeling_qwen3_tts.py`) cho biết chi tiết. Mỗi vị trí là **tổng** của hai embedding: một của luồng chữ,
một của luồng tiếng. Ở chế độ mặc định (theo luồng), vị trí cuối của phần mở đầu là token chữ đầu tiên cộng ký hiệu
"bắt đầu tiếng"; từ đó, mỗi bước sinh nhận tổng 16 embedding của khung vừa sinh **cộng embedding của token chữ kế
tiếp**, rồi "hết chữ" (`tts_eos`), rồi một embedding đệm (`tts_pad`) khi chữ đã hết. Mã cũng có chế độ không theo
luồng (`non_streaming_mode=True`): cả câu chữ đi trước, luồng tiếng ở đó là ký hiệu đệm, rồi mới đến các khung.
Phần mở đầu chung của hai chế độ (vai trong khuôn hội thoại, thẻ ngôn ngữ, vectơ người nói) ta bỏ qua.

Câu của ta: george đọc tám chữ số đầu của π, cách nhau 0,1 s lặng (minh họa), và mỗi từ là một token chữ (minh họa;
bộ tách token của Qwen có thể tách khác).

In [ ]:
SENT = (3, 1, 4, 1, 5, 9, 2, 6)
GEORGE, GAP = SPEAKERS.index("george"), 0.1
used, pieces, spans, t0 = {}, [], [], 0.0
for d in SENT:
    k = used.get(d, 0)
    used[d] = k + 1
    c = int(np.where((digit == d) & (spk == GEORGE) & (idx == k))[0][0])
    x = waves[c] * 0.05 / np.sqrt(np.mean(waves[c] ** 2))  # cùng mức RMS
    pieces += [x, np.zeros(int(GAP * SR))]
    spans.append((t0, t0 + len(x) / SR))
    t0 += len(x) / SR + GAP
sent = np.concatenate(pieces[:-1])
TXT = [WORDS[d] for d in SENT]
N_TXT, N_FR = len(TXT), math.ceil(len(sent) / SR * FPS12)


def word_at(f):
    """Từ đang được nói ở giữa khung f (đếm từ 0); −1 nếu đó là khoảng lặng."""
    t = (f + 0.5) / FPS12
    return next((i for i, (a, b) in enumerate(spans) if a <= t < b), -1)


print(f"câu: {' '.join(TXT)}; {vn(len(sent) / SR, 2)} s → {N_FR} khung 12,5 Hz, {N_TXT} token chữ")
play(sent, "george đọc tám chữ số đầu của π")

Ô dưới xếp câu theo cả hai chế độ thành chuỗi các cặp (luồng chữ, luồng tiếng), rồi dựng đầu vào thật bằng các bảng
embedding nhỏ (8 chiều, khởi tạo ngẫu nhiên: minh họa; mô hình 1,7 tỉ dùng 2.048 chiều) và 16 lớp mã ngẫu nhiên thay
cho token thật. Theo cấu hình phát hành, bảng embedding của lớp 0 có 3.072 dòng (2.048 mã cộng các ký hiệu đặc biệt,
trong đó 2.148 là đệm và 2.149 là "bắt đầu tiếng"), các lớp 1–15 có 2.048 dòng.

In [ ]:
CODEC_PAD, CODEC_BOS = 2148, 2149  # ký hiệu đặc biệt của lớp 0 (cấu hình phát hành)
trailing = TXT[1:] + ["<eos>"]  # token chữ đi cùng các bước sinh: từ thứ 2 trở đi, rồi "hết chữ"
DUAL = [(TXT[0], "<bos>")] + [(trailing[f] if f < len(trailing) else "<pad>", f) for f in range(N_FR)]
SEQ = [(w, "<cpad>") for w in TXT] + [("<eos>", "<cpad>"), ("<pad>", "<bos>")] + [("<pad>", f) for f in range(N_FR)]
print(f"hai luồng: {len(DUAL)} vị trí = 1 + {N_FR} khung; tuần tự: {len(SEQ)} vị trí = {N_TXT} chữ + 2 + {N_FR} khung; "
      f"chênh {len(SEQ) - len(DUAL)}")  # fmt: skip

torch.manual_seed(0)
DIM = 8
VOCAB = sorted(set(TXT)) + ["<eos>", "<pad>"]
E_text = nn.Embedding(len(VOCAB), DIM)
E_codec = nn.ModuleList([nn.Embedding(3072 if q == 0 else CODES12, DIM) for q in range(LAYERS12)])
print(f"bảng embedding lớp 0: {vn(E_codec[0].num_embeddings, 0)} dòng (đệm = {vn(CODEC_PAD, 0)}, bắt đầu tiếng = "
      f"{vn(CODEC_BOS, 0)}); lớp 1–15: {vn(E_codec[1].num_embeddings, 0)} dòng")  # fmt: skip
codes = torch.randint(0, CODES12, (N_FR, LAYERS12), generator=torch.Generator().manual_seed(0))  # mã minh họa


def emb_speech(s):
    if s in ("<bos>", "<cpad>"):
        return E_codec[0](torch.tensor(CODEC_BOS if s == "<bos>" else CODEC_PAD))
    return sum(E_codec[q](codes[s, q]) for q in range(LAYERS12))  # tổng 16 embedding của khung s


with torch.no_grad():
    X_dual = torch.stack([E_text(torch.tensor(VOCAB.index(w))) + emb_speech(s) for w, s in DUAL])
    X_seq = torch.stack([E_text(torch.tensor(VOCAB.index(w))) + emb_speech(s) for w, s in SEQ])
    # cùng phép tính, viết như mã phát hành: tổng theo lớp của cả khung, cộng chữ kế tiếp hoặc đệm
    frames_sum = torch.stack([E_codec[q](codes[:, q]) for q in range(LAYERS12)], 1).sum(1)
    text_next = E_text(torch.tensor([VOCAB.index(trailing[f] if f < len(trailing) else "<pad>") for f in range(N_FR)]))
    assert torch.allclose(X_dual[1:], frames_sum + text_next, atol=1e-5)
print(f"đầu vào: hai luồng {tuple(X_dual.shape)}, tuần tự {tuple(X_seq.shape)}")
assert len(DUAL) == 1 + N_FR and len(SEQ) == N_TXT + 2 + N_FR

Ở chế độ hai luồng, đầu ra ở vị trí 0 là khung 0, và đầu ra ở vị trí f là khung f: khi đoán khung f, mô hình đã đọc
f + 1 token chữ đầu (cho đến khi chữ hết). Ở chế độ tuần tự, khung 0 chỉ được đoán sau khi đã đọc hết cả câu chữ.

### 🤔 Dự đoán trước
Ở chế độ hai luồng, khi đoán khung 5 (giây 0,40–0,48 của câu), mô hình đã đọc bao nhiêu token chữ? Lúc ấy george đang
nói đến từ thứ mấy?

In [ ]:
def read_dual(f):
    """Số token chữ mô hình đã đọc khi đoán khung f, ở chế độ hai luồng."""
    return min(f + 1, N_TXT)


for f in (0, 5, 12, N_FR - 1):
    w = word_at(f)
    said = f"đang nói từ thứ {w + 1} ({TXT[w]})" if w >= 0 else "đang lặng"
    print(
        f"khung {f:2d} ({vn(f / FPS12, 2)}–{vn((f + 1) / FPS12, 2)} s): đã đọc {read_dual(f)} token chữ; george {said}"
    )
started = [sum(a <= (f + 0.5) / FPS12 for a, _ in spans) for f in range(N_FR)]  # số từ đã bắt đầu được nói
lead = [read_dual(f) - started[f] for f in range(N_FR)]
f_max = int(np.argmax(lead))
print(f"luồng chữ đi trước luồng tiếng nhiều nhất {lead[f_max]} từ (ở khung {f_max}); chữ hết từ khung {N_TXT - 1}, "
      f"tiếng còn {N_FR - N_TXT} khung")  # fmt: skip
assert read_dual(5) == 6 and word_at(5) in (0, -1) and max(lead) >= 5

Ở khung 5, mô hình đã đọc 6 token chữ, mà george mới nói đến từ đầu tiên. Quy tắc "mỗi bước một token chữ" đặt luồng
chữ đi trước luồng tiếng khá xa, vì 12,5 khung mỗi giây nhanh hơn nhiều so với tốc độ nói từ. Mô hình phải tự học
xem mình đang đọc đến từ nào; theo báo cáo, giai đoạn S1 dựng một ánh xạ đơn điệu từ biểu diễn chữ sang tiếng
(tr. 5).

Hình dưới vẽ hai chế độ theo vị trí trong chuỗi. Hàng "chữ": ô xanh lá là token chữ (ghi chữ số), E là "hết chữ", ô
xám gạch chéo là đệm. Hàng "tiếng": B là "bắt đầu tiếng", ô xám gạch chéo là đệm, mỗi ô màu là một khung và ghi chữ
số george đang nói ở khung ấy (xanh dương và cam xen kẽ theo từ, ô trắng viền xám là lặng). Tam giác đen chỉ vị trí
mà đầu ra của nó là khung 0.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.1))
rows = {"hai luồng · chữ": 0, "hai luồng · tiếng": 1, "tuần tự · chữ": 2.8, "tuần tự · tiếng": 3.8}


def cell(x, y, color, label="", hatch=None, edge="none"):
    ax.add_patch(Rectangle((x, y), 0.92, 0.88, fc=color, ec=edge, lw=0.6 if edge != "none" else 0, hatch=hatch))
    if label:
        ax.text(x + 0.46, y + 0.44, label, ha="center", va="center", fontsize=6, color="white")


for layout, (yt, ys) in ((DUAL, (0, 1)), (SEQ, (2.8, 3.8))):
    for p, (w, s) in enumerate(layout):
        if w in VOCAB[:-2]:
            cell(p, yt, GREEN, str(WORDS.index(w)))
        elif w == "<eos>":
            cell(p, yt, "#333333", "E")
        else:
            cell(p, yt, "#eef0f3", hatch="////", edge=GREY)
        if s == "<bos>":
            cell(p, ys, GREY, "B")
        elif s == "<cpad>":
            cell(p, ys, "#eef0f3", hatch="////", edge=GREY)
        else:
            wi = word_at(s)
            if wi >= 0:
                cell(p, ys, (BLUE, ORANGE)[wi % 2], str(SENT[wi]))
            else:
                cell(p, ys, "white", edge=GREY)
    p0 = 0 if layout is DUAL else N_TXT + 1
    ax.plot(p0 + 0.46, ys + 1.25, "^", color="k", ms=6)
    ax.text(p0 + 1.2, ys + 1.3, "đầu ra: khung 0", fontsize=7, va="center")
ax.set_xlim(-0.5, len(SEQ) + 0.5)
ax.set_ylim(5.3, -0.3)
ax.set_yticks([v + 0.44 for v in rows.values()], list(rows))
ax.set_xlabel("vị trí trong chuỗi (sau phần mở đầu chung)")
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
vn_axes(fig)
plt.tight_layout()
plt.show()

Ở chế độ tuần tự, chuỗi dài hơn 9 vị trí (8 token chữ và "hết chữ"; ở chế độ hai luồng, token chữ đầu đi chung vị
trí với "bắt đầu tiếng"), và khung 0 đứng sau toàn bộ câu chữ. Với một câu tám từ, chênh lệch ấy nhỏ. Nó lớn khi chữ
**đến dần**, ví dụ từ một mô hình ngôn ngữ lớn đang trả lời: chế độ tuần tự phải chờ câu trả lời xong, chế độ hai luồng
chỉ cần chữ đi trước tiếng.

Mã phát hành nhận cả câu chữ một lúc. Nhưng mô hình là nhân quả, và ở chế độ hai luồng khung f chỉ phụ thuộc f + 1
token chữ đầu, nên cùng cách xếp chạy được khi chữ đến dần. Ô dưới mô phỏng điều đó với hai giả định của notebook:
chữ đến đều 30 token mỗi giây (minh họa), và mỗi khung tốn RTF × 80 ms của bản 12 Hz 1,7 tỉ ở 1 yêu cầu (cách đọc RTF
theo khung ở mục 6). Gói đầu là 4 khung (tr. 6); thời gian giải mã gói không tính.

In [ ]:
R_TEXT = 30  # token chữ mỗi giây từ mô hình phía trước (minh họa)
STEP = TABLE2["12Hz-1.7B"][0][5] * 80  # ms mỗi khung (suy từ RTF của Bảng 2)


def frame_times(n_text, n_frames, layout, rate=R_TEXT, step=STEP):
    """Lúc (ms) mỗi khung được sinh xong; token chữ thứ k đến lúc k / rate giây."""
    arrive = 1000 * np.arange(1, n_text + 1) / rate
    t, out = 0.0, []
    for f in range(n_frames):
        need = arrive[min(f, n_text - 1)] if layout == "dual" else arrive[-1]
        t = max(t, need) + step
        out.append(t)
    return np.array(out)


def first_packet_ms(n_text, layout, **kw):
    return frame_times(n_text, PKT12, layout, **kw)[PKT12 - 1]


print(f"mỗi khung {vn(STEP, 1)} ms; chữ đến {R_TEXT} token/s")
for n_text in (N_TXT, 50, 200):
    print(f"{n_text:3d} token chữ: gói đầu sau {vn(first_packet_ms(n_text, 'dual'), 0):>5s} ms (hai luồng) · "
          f"{vn(first_packet_ms(n_text, 'seq'), 0):>5s} ms (tuần tự)")  # fmt: skip
assert first_packet_ms(200, "dual") == first_packet_ms(N_TXT, "dual") < first_packet_ms(N_TXT, "seq")

In [ ]:
NS = np.arange(4, 301)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(NS, [first_packet_ms(n, "seq") for n in NS], color=ORANGE, lw=2, label="tuần tự: chờ hết chữ")
ax.plot(NS, [first_packet_ms(n, "dual") for n in NS], color=BLUE, lw=2, ls="--", label="hai luồng")
ax.axvline(N_TXT, color=GREY, ls=":", lw=1)
ax.text(N_TXT + 3, 6000, "câu của ta", fontsize=8, color=GREY)
ax.set(xlabel="số token chữ của câu trả lời", ylabel="gói âm thanh đầu sau (ms)", yscale="log", ylim=(100, 15000))
ax.legend(fontsize=8, frameon=False, loc="center right")
vn_axes(fig)
plt.tight_layout()
plt.show()

Ở chế độ hai luồng, gói đầu ra sau cùng một khoảng thời gian dù câu trả lời dài bao nhiêu: chỉ cần 4 token chữ đầu.
Ở chế độ tuần tự, thời gian ấy tăng theo độ dài câu trả lời. Đây là lý do báo cáo dựng mô hình cho "chữ vào theo
luồng, tiếng ra theo luồng" (tr. 2). Mục 10 thử khi chữ đến chậm hơn 12,5 token mỗi giây.

## 8 · MTP: lớp 0 trước, 15 lớp dư lần lượt trong khung

Mỗi khung 12 Hz có 16 token. Báo cáo chia việc đoán chúng làm hai tầng (tr. 4): mô hình chính đọc đặc trưng gộp của
các lớp mã (ở mục 7: tổng 16 embedding của khung trước, cộng token chữ) và đoán lớp 0; rồi mô-đun **MTP**
(*multi-token prediction*) sinh mọi lớp dư. Trong mã phát hành, MTP là một Transformer nhỏ (5 lớp, rộng 1.024, theo
cấu hình phát hành; mô hình chính có 28 lớp, rộng 2.048 ở bản 1,7 tỉ) nhận trạng thái ẩn của mô hình chính cùng
embedding của lớp 0, rồi **sinh tự hồi quy** đúng `num_code_groups − 1` = 15 token, mỗi token nhìn các lớp trước nó
trong cùng khung:

$$p(\mathbf{c}_t \mid \mathbf{h}_t) = p_\theta\big(c_t^{0} \mid \mathbf{h}_t\big) \prod_{j=1}^{15} p_\phi\big(c_t^{j} \mid \mathbf{h}_t,\ c_t^{0}, \dots, c_t^{j-1}\big)$$

Vì sao không đoán 15 lớp dư cùng lúc bằng 15 đầu ra song song, cho nhanh? Vì trong RVQ, lớp j lượng tử hóa phần dư
còn lại *sau khi* các lớp trước đã chọn mã, nên lớp j phụ thuộc vào chính các mã ấy. Ta đo sự phụ thuộc ấy trên dữ
liệu thật: phổ log-mel 40 dải của các khung có tiếng trong FSDD (khung 10 ms, không phải 80 ms), một RVQ nhỏ 3 lớp ×
16 mã học bằng k-means trên tập huấn luyện, rồi đếm. Với mỗi cách "biết trước", ta lập bảng tần suất trên tập huấn
luyện và tính entropy chéo (bit mỗi token) trên tập kiểm tra.

In [ ]:
CFG = {  # cấu hình phát hành (config.json của Qwen3-TTS-12Hz-1.7B-Base và -0.6B-Base)
    "mô hình chính, 1,7 tỉ": {"lớp": 28, "rộng": 2048},
    "mô hình chính, 0,6 tỉ": {"lớp": 28, "rộng": 1024},
    "MTP (cả hai cỡ)": {"lớp": 5, "rộng": 1024, "token mỗi khung": LAYERS12 - 1},
}
for k, v in CFG.items():
    print(f"{k}: " + ", ".join(f"{a} {vn(b, 0)}" for a, b in v.items()))


def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Các bộ lọc tam giác trên thang mel, (n_fft/2 + 1, n_mels), như Chuyên đề 05."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


HANN, FB40 = torch.hann_window(256), mel_filters(256, 40)


def logmel(x):
    """(B, L) → (B, 40, khung): log₁₀ độ lớn trong 40 dải mel; cửa sổ 32 ms, bước 10 ms."""
    s = torch.stft(x, 256, 80, window=HANN, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ FB40).clamp_min(1e-5).log10().transpose(1, 2)


with torch.no_grad():
    frames, owner, has_prev = [], [], []
    for c, x in enumerate(waves):
        m = ((logmel(torch.tensor(x * 0.1 / np.sqrt(np.mean(x**2)), dtype=torch.float32)[None]) + 5) / 2)[0]
        e = m.mean(0)
        keep = (e > e.max() - 1.5).numpy()  # khung có tiếng: trong 1,5 đơn vị log₁₀ của khung to nhất
        frames.append(m[:, keep].T)
        owner.append(np.full(int(keep.sum()), c))
        has_prev.append(np.r_[False, keep[:-1]][keep])  # khung 10 ms ngay trước cũng có tiếng
    FR = torch.cat(frames)
owner, has_prev = np.concatenate(owner), np.concatenate(has_prev)
is_train = np.isin(owner, TRAIN)
print(f"{vn(len(FR), 0)} khung có tiếng × {FR.shape[1]} dải mel; {vn(has_prev.sum(), 0)} khung có khung trước liền kề")


def kmeans(X, k, iters, g):
    C = X[torch.randperm(len(X), generator=g)[:k]].clone()
    for _ in range(iters):
        a = torch.cdist(X, C).argmin(1)
        C = torch.stack([X[a == j].mean(0) if (a == j).any() else C[j] for j in range(k)])
    return C


def rvq_codes(X, layers=3, k=16, seed=0):
    """RVQ: mỗi lớp là k-means (học trên tập huấn luyện) của phần dư sau các lớp trước."""
    g = torch.Generator().manual_seed(seed)
    res, out = X.clone(), []
    for _ in range(layers):
        C = kmeans(res[torch.tensor(is_train)], k, 20, g)
        a = torch.cdist(res, C).argmin(1)
        out.append(a)
        res = res - C[a]
    return torch.stack(out, 1).numpy()


def cross_bits(codes_, j, ctx, k=16, alpha=0.5):
    """Entropy chéo (bit) của lớp j trên tập kiểm tra khi biết các cột ctx; bảng đếm (cộng α) từ tập huấn luyện.
    Cột c < số lớp là lớp c của khung này; cột "prev" là lớp j của khung 10 ms ngay trước."""
    cols = np.c_[codes_, np.r_[codes_[:1, j], codes_[:-1, j]]]  # cột cuối: lớp j của khung trước
    cols = [cols[:, -1] if c == "prev" else cols[:, c] for c in ctx]
    key = sum(col * k**i for i, col in enumerate(cols)) if cols else np.zeros(len(codes_), int)
    trn, tst = is_train & has_prev, ~is_train & has_prev
    T = np.full((k ** len(cols), k), alpha)
    np.add.at(T, (key[trn], codes_[trn, j]), 1)
    T /= T.sum(1, keepdims=True)
    return float(-np.log2(T[key[tst], codes_[tst, j]]).mean())


CODES3 = rvq_codes(FR)
print("RVQ 3 lớp × 16 mã; mã của 5 khung đầu:", CODES3[:5].tolist())

### 🤔 Dự đoán trước
Mỗi lớp có 16 mã, tức tối đa 4 bit. Đã biết lớp 1 của khung, biết thêm lớp 2 có giúp đoán lớp 3 không: gần như không,
hay đáng kể?

In [ ]:
H = {
    ("lớp 2", "không biết gì"): cross_bits(CODES3, 1, []),
    ("lớp 2", "biết lớp 1"): cross_bits(CODES3, 1, [0]),
    ("lớp 3", "không biết gì"): cross_bits(CODES3, 2, []),
    ("lớp 3", "biết lớp 1"): cross_bits(CODES3, 2, [0]),
    ("lớp 3", "biết lớp 1 và 2"): cross_bits(CODES3, 2, [0, 1]),
}
HP = {  # thêm lớp 3 của khung trước: một phần của thứ mà trạng thái ẩn của mô hình chính biết về quá khứ
    "biết lớp 1": cross_bits(CODES3, 2, ["prev", 0]),
    "biết lớp 1 và 2": cross_bits(CODES3, 2, ["prev", 0, 1]),
}
for (layer, ctx), v in H.items():
    print(f"{layer}, {ctx:16s}: {vn(v, 3)} bit")
gap = H["lớp 3", "biết lớp 1"] - H["lớp 3", "biết lớp 1 và 2"]
gap_prev = HP["biết lớp 1"] - HP["biết lớp 1 và 2"]
print(f"biết thêm lớp 2 bớt {vn(gap, 3)} bit cho lớp 3")
print(f"khi đã biết cả lớp 3 của khung trước: {vn(HP['biết lớp 1'], 3)} → {vn(HP['biết lớp 1 và 2'], 3)} bit "
      f"(bớt {vn(gap_prev, 3)})")  # fmt: skip
assert gap > 0.3 and gap_prev > 0.1, "trong một khung, lớp 3 phụ thuộc lớp 2 ngay cả khi đã biết lớp 1"
assert H["lớp 2", "biết lớp 1"] < H["lớp 2", "không biết gì"] - 0.15

Đáng kể. Biết lớp 1, lớp 3 còn 3,884 bit; biết thêm lớp 2, chỉ còn 3,483 bit. Một bộ đoán song song (mọi lớp dư cùng
lúc, từ cùng một trạng thái) phải chọn lớp 3 mà không biết lớp 2 đã chọn mã nào, nên trên tập kiểm tra nó tốn thêm
chừng ấy bit cho mỗi khung, chỉ riêng ở cặp lớp này. Phần chênh vẫn còn (nhỏ hơn) khi biết cả lớp 3 của khung trước,
tức một phần lịch sử mà trạng thái ẩn của mô hình chính có. MTP sinh lần lượt, nên lớp j nhìn được mọi lớp trước nó
trong khung. Hai lưu ý: đây là số bit, không phải chất lượng âm thanh; và RVQ của ta (3 lớp × 16 mã trên khung 10 ms
của phổ mel) nhỏ hơn rất nhiều so với 16 lớp × 2.048 mã trên khung 80 ms của sóng âm.

Hình dưới vẽ các entropy chéo ấy: mỗi cột là một cách "biết trước", cột gạch chéo là khi biết thêm lớp 3 của khung
trước; vạch đứt là 4 bit, mức của một mã đoán đều.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
labels = [f"{a}\n{b}" for a, b in H] + ["lớp 3\nbiết khung trước\n+ lớp 1", "lớp 3\nbiết khung trước\n+ lớp 1 và 2"]
vals = [*H.values(), *HP.values()]
cols = [GREY, BLUE, GREY, BLUE, GREEN, BLUE, GREEN]
for i, (v, col) in enumerate(zip(vals, cols, strict=True)):
    ax.bar(i, v, color=col, hatch="//" if i >= 5 else None, ec="white", width=0.7)
    ax.text(i, v - 0.12, vn(v, 2), ha="center", va="top", fontsize=8, color="white", fontweight="bold")
ax.axhline(4, color="k", ls="--", lw=0.8)
ax.text(6.45, 4.05, "4 bit: 16 mã đều nhau", ha="right", va="bottom", fontsize=7)
ax.set_xticks(range(len(vals)), labels, fontsize=7)
ax.set(ylabel="bit mỗi token (tập kiểm tra)", ylim=(0, 4.4))
vn_axes(fig)
plt.tight_layout()
plt.show()

### Đếm bước: phẳng, delay pattern, MTP

Chuyên đề 07 so hai cách cho 32 lớp: delay pattern và Local Transformer. Với 16 lớp, có ba cách:
- **phẳng**: mỗi token một bước của mô hình chính, 16 · T bước cho T khung;
- **delay pattern** (mô hình chính 8 tỉ của MOSS-TTS): lớp j dịch đi j bước, T + 15 bước;
- **MTP** (Qwen3-TTS 12 Hz, giống Local Transformer của MOSS-TTS): T bước của mô hình chính và 15 · T bước nhỏ.

Ta dùng token thật: 6 chữ số do mô hình đọc nhỏ của Chuyên đề 05 sinh ra (lưu trong `codec-weights.json`), lấy 16 lớp
đầu trong 32 lớp.

In [ ]:
SH = W["show"]["0"]
assert SH["k"][0] == 32
A16 = torch.cat([torch.tensor(c)[:, :LAYERS12] for c in SH["codes"][0]])  # (khung, 16 lớp)
T6 = len(SH["codes"][0][0])
print(
    f"{len(SH['codes'][0])} chữ số {SH['target']} → {A16.shape[0]} khung × {A16.shape[1]} lớp; chữ số đầu có {T6} khung"
)


def delay(A):
    """(T, nq) → (T + nq − 1, nq): lớp j dịch đi j bước; −1 là ô trống."""
    T_, nq = A.shape
    D_ = torch.full((T_ + nq - 1, nq), -1)
    for j in range(nq):
        D_[j : j + T_, j] = A[:, j]
    return D_


D16 = delay(A16)
assert torch.equal(torch.stack([D16[j : j + len(A16), j] for j in range(LAYERS12)], 1), A16)


def steps(T, nq=LAYERS12):
    """(bước mô hình chính, bước nhỏ, bước mô hình chính cho đến khi khung 0 đủ nq token)."""
    return {"phẳng": (T * nq, 0, nq), "delay": (T + nq - 1, 0, nq), "MTP": (T, T * (nq - 1), 1)}


for T in (T6, len(A16), 75, 125):
    print(f"{T:3d} khung ({vn(T / FPS12, 2)} s): " + " · ".join(
        f"{k} {vn(a, 0)} bước lớn" + (f" + {vn(b, 0)} bước nhỏ" if b else "") for k, (a, b, _) in steps(T).items()))  # fmt: skip
print("khung 0 đủ 16 token sau: " + ", ".join(f"{k} {c} bước lớn" for k, (_, _, c) in steps(1).items()))
assert D16.shape == (len(A16) + 15, 16) and steps(125)["MTP"] == (125, 1875, 1) and steps(125)["phẳng"][0] == 2000
assert steps(1)["delay"][2] == 16

Hình dưới tô mỗi token (lớp × khung) của chữ số đầu theo bước của mô hình chính sinh ra nó, và ghi số bước ấy. Ở cách
phẳng và delay pattern, mỗi lớp của khung 0 cần một bước lớn riêng; ở MTP, cả 16 lớp của khung t ra ở bước lớn t,
15 lớp dư do 15 bước nhỏ của MTP sinh lần lượt.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4.2), sharey=True)
order = {
    "phẳng: 16 · T bước": np.arange(T6)[None, :] * LAYERS12 + np.arange(LAYERS12)[:, None],
    "delay pattern: T + 15 bước": np.arange(T6)[None, :] + np.arange(LAYERS12)[:, None],
    "MTP: T bước lớn (+ 15 bước nhỏ mỗi khung)": np.broadcast_to(np.arange(T6)[None, :], (LAYERS12, T6)),
}
for ax, (title, S) in zip(axes, order.items(), strict=True):
    ax.imshow(S, aspect="auto", cmap="viridis", interpolation="nearest")
    for q in range(LAYERS12):
        for t in range(T6):
            ax.text(t, q, str(S[q, t]), ha="center", va="center", fontsize=6,
                    color="white" if S[q, t] < S.max() * 0.6 else "k")  # fmt: skip
    ax.set_title(title, fontsize=8)
    ax.set_xticks(range(T6), [str(t) for t in range(T6)])
    ax.set_xlabel("khung")
axes[0].set_yticks([0, 5, 10, 15], ["lớp 0", "lớp 5", "lớp 10", "lớp 15"])
plt.tight_layout()
plt.show()

Với một câu 10 giây (125 khung), cách phẳng cần 2.000 bước lớn; delay pattern chỉ 140, nhưng khung 0 chỉ đủ 16 token
sau 16 bước; MTP cần 125 bước lớn và 1.875 bước nhỏ, và khung 0 đủ 16 token ngay sau bước lớn đầu tiên. Đó là câu
của báo cáo: MTP cho phép "giải mã ngay từ khung đầu tiên" (tr. 2). Mỗi bước nhỏ chạy một mạng nhỏ hơn nhiều so với
mô hình chính, nhưng 15 bước ấy vẫn nối tiếp nhau trong từng khung.

## 9 · Hậu huấn luyện: tỉ số quan trọng trên chuỗi dài

Sau ba giai đoạn tiền huấn luyện, báo cáo hậu huấn luyện ba bước (tr. 5): **DPO** trên các cặp ưa thích dựng từ phản
hồi của người cho tiếng nói nhiều ngôn ngữ; **GSPO** với phần thưởng theo luật; rồi tinh chỉnh nhẹ theo từng giọng trên
mô hình gốc. DPO đã có ở [Chuyên đề 04](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rlhf/) (mục 10) và
[Bài 26](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/). Ở đây ta xem chỗ khác nhau của GSPO.

GRPO (DeepSeekMath, 2024) và GSPO (Zheng và cộng sự, 2025, cùng nhóm Qwen) đều lấy một nhóm câu trả lời từ chính sách
cũ $\pi_{\text{old}}$, chấm điểm từng câu, rồi cập nhật chính sách mới $\pi_\theta$. Chúng khác nhau ở **tỉ số quan
trọng**, thứ sửa cho việc mẫu đến từ $\pi_{\text{old}}$ chứ không phải $\pi_\theta$:

$$w_t = \frac{\pi_\theta(y_t \mid x, y_{<t})}{\pi_{\text{old}}(y_t \mid x, y_{<t})}\ \ \text{(GRPO: mỗi token một tỉ số)}, \qquad s = \Big(\frac{\pi_\theta(y \mid x)}{\pi_{\text{old}}(y \mid x)}\Big)^{1/|y|} = \exp\Big(\frac{1}{|y|}\sum_t \log w_t\Big)\ \ \text{(GSPO)}$$

Tỉ số của cả câu là tích $\prod_t w_t$. GSPO lấy căn bậc $|y|$ của tích ấy (trung bình nhân của các $w_t$) và cắt theo
cả câu thay vì theo từng token. Báo cáo Qwen3-TTS không nói vì sao chọn GSPO. Bài báo GSPO lập luận rằng tỉ số của
từng token chỉ dựa trên một mẫu nên chủ yếu là nhiễu, nhiễu ấy tích lại theo độ dài câu trả lời, và đơn vị tối ưu nên
trùng với đơn vị của phần thưởng, tức cả câu. Một câu nói rất dài nếu tính theo token: 10 giây tiếng 12 Hz là 125
khung, tức 2.000 token nếu tính cả 16 lớp (báo cáo không nói GSPO tính trên những lớp nào).

Thí nghiệm đồ chơi: mỗi vị trí có một phân phối trên 2.048 mã (logit ngẫu nhiên), chính sách mới là chính sách cũ cộng
một nhiễu nhỏ ở logit (như sau một bước cập nhật), và các vị trí độc lập với nhau. Mọi thứ ở đây là minh họa. Ta lấy
400 chuỗi dài 2.000 token từ chính sách cũ, rồi đo ba loại tỉ số theo độ dài L.

In [ ]:
EPS_GRPO = (0.2, 0.27)  # khoảng cắt trái/phải của GRPO trong thí nghiệm của bài báo GSPO (mục 5.1)
LMAX, NSEQ = 2000, 400


def toy_ratios(sigma, seed=0):
    """log w_t của NSEQ chuỗi dài LMAX lấy từ chính sách cũ; chính sách mới = cũ + nhiễu σ ở logit."""
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(LMAX, CODES12, generator=g) * 2
    lp_old = torch.log_softmax(z, -1)
    lp_new = torch.log_softmax(z + sigma * torch.randn(LMAX, CODES12, generator=g), -1)
    y = torch.multinomial(lp_old.exp(), NSEQ, replacement=True, generator=g)  # (vị trí, chuỗi)
    kl = float((lp_old.exp() * (lp_old - lp_new)).sum(-1).mean())
    return (lp_new.gather(1, y) - lp_old.gather(1, y)).T.numpy(), kl


def short(v):
    return sci(v) if v < 0.01 else vn(v, 3)


def bands(logw, L):
    """Phân vị 5%, 50%, 95% của tích các tỉ số, của tỉ số GSPO, và phần token GRPO rơi ra ngoài khoảng cắt."""
    lp = logw[:, :L].sum(1)
    prod, gspo = np.exp(np.percentile(lp, [5, 50, 95])), np.exp(np.percentile(lp / L, [5, 50, 95]))
    w = np.exp(logw[:, :L])
    out = float(((w < 1 - EPS_GRPO[0]) | (w > 1 + EPS_GRPO[1])).mean())
    return prod, gspo, np.exp(np.percentile(logw[:, :L], [5, 95])), out


LOGW, KL = toy_ratios(0.1)
print(f"KL trung bình mỗi token {sci(KL)} nat; độ lệch chuẩn của log w_t: {vn(LOGW.std(), 3)}")
print(f"khoảng cắt của GRPO: [{vn(1 - EPS_GRPO[0], 2)}; {vn(1 + EPS_GRPO[1], 2)}]; mọi khoảng dưới là phân vị 5%–95%")
for L in (1, 16, 125, 2000):
    prod, gspo, tok, out = bands(LOGW, L)
    print(f"L = {vn(L, 0):>5s}: tích ∏w_t {short(prod[0])} … {short(prod[2])} (trung vị {short(prod[1])}) · "
          f"GSPO s {vn(gspo[0], 4)} … {vn(gspo[2], 4)} · từng token {vn(tok[0], 3)} … {vn(tok[1], 3)}, "
          f"ngoài khoảng cắt {pct(out)}")  # fmt: skip
p16, g16, _, o16 = bands(LOGW, 16)
p2k, g2k, _, o2k = bands(LOGW, 2000)
print(f"L = 2.000: tích trải {vn(math.log10(p2k[2] / p2k[0]), 1)} bậc độ lớn; GRPO cắt khoảng "
      f"{round(o2k * 2000)} token mỗi chuỗi")  # fmt: skip
assert p2k[2] / p2k[0] > 1e5 > p16[2] / p16[0] and g2k[2] - g2k[0] < 0.01 < g16[2] - g16[0]
assert 0.01 < o16 < 0.04 and 0.01 < o2k < 0.04, "phần token bị cắt gần như không đổi theo độ dài"

Hình dưới vẽ khoảng từ phân vị 5% đến 95% theo độ dài chuỗi (trục ngang theo thang log). Bên trái (trục dọc cũng theo
thang log): tích của mọi tỉ số token. Bên phải (trục dọc thường): tỉ số GSPO (xanh dương) và tỉ số của từng token mà
GRPO dùng (xanh lá, gạch chéo), cùng hai vạch của khoảng cắt GRPO. Vạch chấm là 125 token (chỉ lớp 0 của câu 10 giây)
và 2.000 token (cả 16 lớp).

In [ ]:
LS = np.unique(np.round(np.logspace(0, np.log10(LMAX), 40)).astype(int))
B = [bands(LOGW, L) for L in LS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.fill_between(LS, [b[0][0] for b in B], [b[0][2] for b in B], color=ORANGE, alpha=0.4, lw=0, label="tích ∏ w_t")
a1.plot(LS, [b[0][1] for b in B], color=ORANGE, lw=1.5, label="trung vị")
a1.set(yscale="log", ylim=(1e-10, 10), ylabel="tỉ số quan trọng", title="tích của mọi tỉ số token")
a2.fill_between(LS, [b[2][0] for b in B], [b[2][1] for b in B], color=GREEN, alpha=0.3, hatch="//", lw=0,
                label="GRPO: tỉ số của từng token")  # fmt: skip
a2.fill_between(LS, [b[1][0] for b in B], [b[1][2] for b in B], color=BLUE, alpha=0.6, lw=0, label="GSPO: s")
for v in (1 - EPS_GRPO[0], 1 + EPS_GRPO[1]):
    a2.axhline(v, color=GREEN, ls="--", lw=0.8)
a2.text(1.1, 1 + EPS_GRPO[1] + 0.005, "khoảng cắt của GRPO", fontsize=7, color=GREEN, va="bottom")
a2.set(ylim=(0.6, 1.33), title="GSPO và GRPO")
for ax in (a1, a2):
    ax.set(xscale="log", xlabel="độ dài chuỗi L (token)")
    ax.axhline(1, color="k", lw=0.6)
    for L, txt in ((125, "125"), (2000, "2.000")):
        ax.axvline(L, color=GREY, ls=":", lw=1)
        ax.text(L * 0.93, ax.get_ylim()[1], txt, ha="right", va="top", fontsize=8, color=GREY)
    ax.legend(fontsize=8, frameon=False, loc="lower left")
vn_axes(fig)
plt.tight_layout()
plt.show()

Ở L = 2.000, tích các tỉ số trải 6,4 bậc độ lớn giữa phân vị 5% và 95%, và trung vị của nó rất nhỏ: dùng nó làm
trọng số thì vài chuỗi hiếm gánh gần hết. Tỉ số GSPO thì hẹp dần khi chuỗi dài ra, vì trung bình của nhiều log w_t ít
nhiễu hơn từng cái. Tỉ số của từng token mà GRPO dùng có cùng độ trải ở mọi độ dài, và một phần token gần như không đổi
(khoảng 2%) rơi ra ngoài khoảng cắt: với 2.000 token là vài chục token mỗi chuỗi, và token nào bị cắt là do nhiễu chứ
không do cả câu tốt hay dở. Vì s hẹp như vậy, bài báo GSPO (mục 5.1) dùng cho nó khoảng cắt cỡ 3·10⁻⁴ và 4·10⁻⁴, nhỏ
hơn khoảng cắt của GRPO hàng trăm lần. Đồ chơi này chỉ cho thấy độ trải của các tỉ số theo độ dài; nó không cho thấy
GSPO huấn luyện một mô hình TTS tốt hơn.

## 10 · Thử thay đổi

### Khối của bộ giải mã 25 Hz
Khối nhỏ hơn thì gói đầu đến sớm hơn, nhưng BigVGAN vẫn cần 130 ms phổ mel phía sau. Nếu một khối ngắn hơn 130 ms, gói
đầu phải đợi thêm khối. Sửa `CHUNKS` hay `AHEADS` rồi chạy lại.

In [ ]:
CHUNKS, AHEADS = (2, 4, 8, 16), (0, 1, 2)


def first_audio(chunk, ahead):
    """(token phải chờ, ms âm thanh của gói đầu): gói đầu là khối mel i đầu tiên vượt quá 130 ms."""
    i = next(i for i in range(99) if (i + 1) * chunk * MS25 > VOC_MS)
    return need_tokens(i, chunk=chunk, ahead=ahead), (i + 1) * chunk * MS25 - VOC_MS


for chunk in CHUNKS:
    cells = [first_audio(chunk, a) for a in AHEADS]
    print(f"khối {chunk:2d} token ({vn(chunk * MS25, 0):>3s} ms): " + " · ".join(
        f"nhìn trước {a}: chờ {w_:2d} token, gói đầu {vn(au, 0):>3s} ms" for a, (w_, au) in zip(AHEADS, cells, strict=True)))  # fmt: skip
assert first_audio(8, 1) == (16, 190) and first_audio(4, 1) == (8, 30)

Khối 4 token chỉ bắt chờ 8 token, nhưng gói đầu chỉ có 30 ms âm thanh; khối 2 token (80 ms) còn ngắn hơn phần BigVGAN
giữ lại, nên gói đầu phải đợi đến khối thứ hai. Khối 16 token cho gói đầu 510 ms nhưng phải chờ 32 token. Mỗi khối
nhìn trước thêm làm mô hình chờ thêm một khối token, mà không đổi lượng âm thanh của gói đầu.

### Chữ đến chậm
Ở chế độ hai luồng, mỗi khung cần một token chữ mới cho đến khi chữ hết, tức 12,5 token mỗi giây tiếng nói. Nếu mô
hình phía trước viết chậm hơn thế, tiếng phải ngừng chờ chữ. Ô dưới dùng một câu trả lời 200 token, mỗi token ứng với
số khung tiếng trung bình của câu của ta (minh họa), và đếm thời gian phải ngừng sau khi đã bắt đầu phát.

In [ ]:
N_LONG = 200
FR_PER_TOK = round(N_FR / N_TXT)
print(f"{N_LONG} token chữ × {FR_PER_TOK} khung mỗi token (câu của ta: {N_FR} khung cho {N_TXT} token)")
for rate in (5, 10, 12.5, 30, 100):
    ft = frame_times(N_LONG, N_LONG * FR_PER_TOK, "dual", rate=rate)
    arrive = ft[PKT12 - 1 :: PKT12]
    _, stall = schedule(arrive, np.full(len(arrive), PKT12 * 80.0))
    print(f"chữ đến {vn(rate, 1).removesuffix(',0'):>4s} token/s: gói đầu sau {vn(arrive[0], 0):>4s} ms, "
          f"phải ngừng tổng cộng {vn(stall / 1000, 1)} s trong {vn(len(ft) / FPS12, 0)} s tiếng")  # fmt: skip

Từ 12,5 token mỗi giây trở lên, tiếng không phải ngừng; dưới mức ấy thì có, vì trong lúc chữ còn đến, quy tắc của
mã phát hành đọc đúng một token chữ cho mỗi khung. Mô hình phía trước viết càng chậm, gói đầu cũng đến càng muộn.

### RVQ nhỏ hơn
Với 8 mã mỗi lớp thay vì 16, lớp 3 còn phụ thuộc lớp 2 nhiều không?

In [ ]:
CODES_K8 = rvq_codes(FR, k=8)
h1, h12 = cross_bits(CODES_K8, 2, [0], k=8), cross_bits(CODES_K8, 2, [0, 1], k=8)
print(f"8 mã mỗi lớp (tối đa 3 bit): lớp 3 biết lớp 1 {vn(h1, 3)} bit, biết lớp 1 và 2 {vn(h12, 3)} bit "
      f"(bớt {vn(h1 - h12, 3)}; với 16 mã: bớt {vn(gap, 3)})")  # fmt: skip
assert h1 - h12 > 0.1

Với 8 mã, phần chênh nhỏ hơn nhưng vẫn còn: sự phụ thuộc giữa các lớp trong khung không phải do riêng một cấu hình.

### Bước cập nhật nhỏ hơn
Với nhiễu σ = 0,03 ở logit thay vì 0,1, tích các tỉ số trên 2.000 token còn trải rộng không?

In [ ]:
LOGW3, KL3 = toy_ratios(0.03)
prod, gspo, tok, out = bands(LOGW3, 2000)
print(f"σ = 0,03 (KL mỗi token {sci(KL3)} nat), L = 2.000: tích {short(prod[0])} … {short(prod[2])} "
      f"(trải {vn(math.log10(prod[2] / prod[0]), 1)} bậc) · GSPO s {vn(gspo[0], 4)} … {vn(gspo[2], 4)} · "
      f"token ngoài khoảng cắt {pct(out, 2)}")  # fmt: skip

Với bước cập nhật nhỏ hơn, tích hẹp lại nhưng vẫn trải gần hai bậc độ lớn, trong khi gần như không token nào chạm
khoảng cắt của GRPO.

## 11 · Giới hạn

- **Không mô hình lớn nào chạy ở đây.** Mô hình ngôn ngữ Qwen3, bộ mã hóa của Qwen2-Audio, DiT, BigVGAN, WavLM, bộ mã
  hóa người nói và MTP thật: ta chỉ dựng lại các phép tính xung quanh chúng, bằng các con số của báo cáo và của mã phát
  hành.
- **Số đo từng lớp ở mục 4 là của Chuyên đề 05**, trên một bộ token hóa thu nhỏ (32 lớp × 1.024 mã, giám sát ngữ nghĩa
  kiểu ASR), không phải bộ 12 Hz của Qwen.
- **Hai cách đọc ở mục 6 là của notebook:** RTF của bản 12 Hz tính theo khung, và việc mô hình ngôn ngữ với bộ giải mã
  chạy nối tiếp hay gối đầu. Báo cáo không nói rõ cả hai.
- **Mục 7:** mã phát hành nhận cả câu chữ một lúc. Chữ đến dần, tốc độ của mô hình phía trước, một từ một token và
  thời gian mỗi khung là giả định của notebook.
- **Mục 8** đo số bit trên một RVQ nhỏ của phổ mel, không đo chất lượng âm thanh. **Mục 9** là một đồ chơi với các vị
  trí độc lập; nó cho thấy độ trải của các tỉ số, không cho thấy thuật toán nào huấn luyện tốt hơn.
- **Những gì báo cáo không công bố** thì notebook cũng không có: hỗn hợp dữ liệu, kích thước bộ token hóa và bộ mã hóa
  người nói, luật thưởng của GSPO, số cặp của DPO, độ dài từng giai đoạn huấn luyện.

## Tóm tắt

- **Hai bộ token hóa:** 25 Hz × 1 bảng 32.768 mã = 375 bit/s, mô hình chính chạy 25 bước mỗi giây tiếng nói;
  12,5 Hz × 16 lớp × 2.048 mã = 2.200 bit/s, mô hình chính chỉ chạy 12,5 bước mỗi giây, MTP sinh 187,5 token còn lại.
  10 phút tiếng nói là 15.000 hay 7.500 vị trí; giai đoạn S3 mở chuỗi từ 8.192 lên 32.768.
- **Lớp đầu mang nội dung** (số đo của Chuyên đề 05): một lớp đã cho 81,9% chữ số đúng nhưng chỉ 72,0% giọng; từ 1
  lên 4 lớp, giọng tăng 18,6 điểm, chữ số 7,0 điểm. Bỏ giám sát ngữ nghĩa, lớp đầu chỉ còn 71,9%.
- **Giải mã 25 Hz theo khối:** khối 8 token, nhìn lại 3 khối và nhìn trước 1 khối; phải chờ 16 token, gói đầu chỉ có
  190 ms âm thanh, phần đi trước luôn là 450 ms. Danh sách ở tr. 3 là 5 khối chứ không phải 4. Bộ 12 Hz không cần
  ngữ cảnh tương lai.
- **Độ trễ:** 12 dòng của Bảng 2 đều là TTFP + giải mã; 93 + 4 = 97 ms cho bản 12 Hz 0,6 tỉ. Năm trong sáu RTF của bản
  12 Hz khớp (TPP + giải mã)/80 ms, như thể TPP là thời gian cho một khung chứ không phải một gói (cách đọc của
  notebook). Phát lại không phải chờ ở trạng thái ổn định; riêng bản 25 Hz ở 6 yêu cầu có thể ngừng 42 ms sau gói đầu
  nếu hai khâu chạy nối tiếp.
- **Hai luồng:** mỗi bước nhận tổng 16 embedding của khung trước cộng token chữ kế tiếp (rồi đệm khi chữ hết); khung 0
  chỉ cần một token chữ, nên gói đầu không phụ thuộc độ dài câu trả lời, miễn chữ đến không chậm hơn 12,5 token mỗi
  giây.
- **MTP:** biết lớp 2, lớp 3 bớt 0,402 bit (3,884 → 3,483), thứ một bộ đoán song song bỏ lỡ; khung 0 đủ 16 token sau
  1 bước lớn, so với 16 của delay pattern.
- **GSPO:** trên 2.000 token, tích các tỉ số token trải 6,4 bậc độ lớn, còn trung bình nhân của GSPO nằm trong
  0,9916–0,9990.

## Câu hỏi ôn tập

1. Một câu 6 giây: mô hình chính của bản 25 Hz và của bản 12 Hz chạy bao nhiêu bước? MTP chạy bao nhiêu bước nhỏ?
2. Nếu khối của bộ giải mã 25 Hz là 4 token và vẫn nhìn trước 1 khối, mô hình phải sinh bao nhiêu token trước gói đầu,
   và gói ấy có bao nhiêu ms âm thanh?
3. Vì sao độ trễ gói đầu của bản 25 Hz tăng nhanh hơn bản 12 Hz khi số yêu cầu đồng thời tăng từ 1 lên 6?
4. Ở chế độ hai luồng, khi chữ đã hết thì luồng chữ chứa gì? Vì sao chế độ tuần tự không thể nói ngay khi chữ còn đang
   đến?
5. Một bộ đoán song song cho 15 lớp dư thiếu thông tin gì? Mục 8 đo phần thiếu ấy bằng phép tính nào?
6. Vì sao tích các tỉ số của 2.000 token không dùng được làm trọng số, và GSPO làm gì thay thế?

<details><summary>Gợi ý đáp án</summary>

1. 6 · 25 = 150 bước cho bản 25 Hz; 6 · 12,5 = 75 bước lớn cho bản 12 Hz, cộng 75 · 15 = 1.125 bước nhỏ của MTP.
2. Khối 0 cần khối 1, nên phải chờ 8 token; khối 4 token là 160 ms phổ mel, trừ 130 ms mà BigVGAN giữ lại, gói đầu chỉ
   có 30 ms âm thanh (mục 10).
3. Phần giải mã của bản 25 Hz (DiT rồi BigVGAN) tăng từ 25 lên 147 ms, còn ConvNet nhân quả của bản 12 Hz chỉ
   từ 4 lên 5 ms; phần của mô hình ngôn ngữ thì tăng ở cả hai.
4. Embedding đệm `tts_pad` (sau một bước "hết chữ" `tts_eos`). Ở chế độ tuần tự, khung 0 chỉ được đoán sau khi mô hình đã
   đọc hết mọi token chữ, nên nó phải chờ câu chữ xong.
5. Nó không biết các lớp trước đã chọn mã nào trong cùng khung, trong khi ở RVQ mỗi lớp lượng tử hóa phần dư sau các lớp
   ấy. Mục 8 so entropy chéo của lớp 3 khi biết lớp 1 với khi biết lớp 1 và 2: phần chênh là số bit bộ đoán song song
   bỏ lỡ.
6. Log của tích là tổng 2.000 số nhiễu, nên độ trải của nó lớn dần theo căn bậc hai của độ dài, và tích trải 6,4 bậc
   độ lớn: vài chuỗi gánh gần hết trọng số. GSPO dùng trung bình nhân (căn bậc |y| của tích), hẹp dần khi chuỗi dài
   ra, và cắt theo cả câu.
</details>

## Tìm hiểu thêm

- Qwen Team (2026), *Qwen3-TTS Technical Report*: [arXiv:2601.15621](https://arxiv.org/abs/2601.15621); mã và mô hình:
  [QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS).
- Chu và cộng sự (2024), *Qwen2-Audio Technical Report*: [arXiv:2407.10759](https://arxiv.org/abs/2407.10759).
- Défossez và cộng sự (2024), *Moshi* (bộ token hóa Mimi): [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Chen và cộng sự (2022), *WavLM*: [arXiv:2110.13900](https://arxiv.org/abs/2110.13900).
- Lipman và cộng sự (2023), *Flow Matching for Generative Modeling*: [arXiv:2210.02747](https://arxiv.org/abs/2210.02747);
  Peebles & Xie (2023), *DiT*: [arXiv:2212.09748](https://arxiv.org/abs/2212.09748); Lee và cộng sự (2023), *BigVGAN*:
  [arXiv:2206.04658](https://arxiv.org/abs/2206.04658).
- Rafailov và cộng sự (2023), *DPO*: [arXiv:2305.18290](https://arxiv.org/abs/2305.18290); Shao và cộng sự (2024),
  *DeepSeekMath* (GRPO): [arXiv:2402.03300](https://arxiv.org/abs/2402.03300); Zheng và cộng sự (2025), *GSPO*:
  [arXiv:2507.18071](https://arxiv.org/abs/2507.18071).
- Alibaba Token Foundry (2026), *Qwen-Audio-3.0-TTS*, một hệ thống TTS khác của Alibaba có tiếng Việt:
  [arXiv:2607.23938](https://arxiv.org/abs/2607.23938).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Quay lại:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) ·
[Chuyên đề 05 · Token hóa âm thanh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/) ·
[Chuyên đề 07 · MOSS-TTS](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/moss-tts/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)